<a href="https://colab.research.google.com/github/Jonchyk/Datamgmt/blob/main/Capstone_Workbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Set Up

In [244]:
#---------------------------SETUP----------------------------------
#get useful libraries
import time, os, sys, re #basics
import zipfile, json, datetime, string   #string for annotating points in scatter
import numpy as np #basic math
# Install tabula-py using pip
!pip install tabula-py
# Import the correct module
from tabula import read_pdf
import statsmodels.api as sm
from statsmodels.formula.api import ols

!pip install pyreadstat


import matplotlib.pyplot as plt #import pylab as plt #apparently discouraged now:
 #https://stackoverflow.com/questions/11469336/what-is-the-difference-between-pylab-and-pyplot
 #https://www.tutorialspoint.com/matplotlib/matplotlib_pylab_module.htm

import pandas as pd
import pandas_datareader as pdr
from pandas_datareader import wb
from pandas.io.formats.style import Styler
#s4 = Styler(df4, uuid_len=0, cell_ids=False)

import urllib  #weird, guess need to have os and pandas imported for this to work  %TODO/LATER ditch it, its weird anyway, just use wget/curl

from google.colab import files

#import webbrowser

import seaborn as sns

from google.colab import data_table
data_table.enable_dataframe_formatter() #this enables spreadsheet view upon calling dataframe (without() )

#many tricks how to extend notebook functionality
#https://coderzcolumn.com/tutorials/python/list-of-useful-magic-commands-in-jupyter-notebook-lab
#will display all output not just last command
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

#MAGICS and THEMES/STYLES: important! does affect not just shading/colors, but also fonts, spacing, etc
#(even if you only select default (v not selecting anything) [but does seem to work better if you do make explicit sleections])

###magics: https://ipython.readthedocs.io/en/stable/interactive/magics.html
#most essential setup for vis: it does affect vis! careful!! stick with inline, maybe notebook; others mostly for non-notebook, eg spyder environ
#https://jakevdp.github.io/PythonDataScienceHandbook/04.00-introduction-to-matplotlib.html recomends *inline*!
#show current one:
#%matplotlib
#%matplotlib --list
#interactive plots:
#%matplotlib notebook
#static images of your plot:
%matplotlib inline
#may play with this one and other magics (btw default is probably agg)
#%matplotlib nbagg
##https://www.marktechpost.com/2023/10/20/6-magic-commands-for-jupyter-notebooks-in-python-data-science/
#%%latex
#%ai
#%run
#%writefile
#%history -n

###themes/styles: https://matplotlib.org/stable/gallery/style_sheets/style_sheets_reference.html
#https://jakevdp.github.io/PythonDataScienceHandbook/04.11-settings-and-stylesheets.html
#https://matplotlib.org/stable/tutorials/introductory/customizing.html
#here more about art and style than under the hood functionality as with magics, explore and experiment
#many may find 'default' or seaborn ones more pleasing; my fav 'classic' is back from 90s ;)
#plt.style.available #list available styles :) may install more
#plt.style.use('default') # more delicate subtle than classic
plt.style.use('classic')  #  'seaborn-whitegrid' 'seaborn-white' 'seaborn-poster'
# btw: magics v theme/style sequence matters, eg if i specify classic style before inline magic, i wouldnt get grey bounding box im getting

#sometimes have to install library which you get from https://pypi.org/
#!pip install geopandas

In [245]:
import plotly.express as px
from statsmodels.stats.multicomp import pairwise_tukeyhsd



#Datasets



##Ministry of Science and Education Report - Population and Ethnic Demographics by Region
Report accessed here, table with population information summarized and rebuilt using Tabula. All data is from 2022

 https://stat.gov.kg/media/publicationarchive/fa3392ca-c76e-456f-badd-7a324ff5204d.pdf

In [246]:
!wget --no-check-certificate "https://drive.google.com/uc?export=download&id=1fTVSC1y_1MSfCS44n7hR-Z7eU-vLtLTn" -O moesreport.pdf


--2025-03-29 18:28:06--  https://drive.google.com/uc?export=download&id=1fTVSC1y_1MSfCS44n7hR-Z7eU-vLtLTn
Resolving drive.google.com (drive.google.com)... 74.125.126.102, 74.125.126.139, 74.125.126.113, ...
Connecting to drive.google.com (drive.google.com)|74.125.126.102|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1fTVSC1y_1MSfCS44n7hR-Z7eU-vLtLTn&export=download [following]
--2025-03-29 18:28:06--  https://drive.usercontent.google.com/download?id=1fTVSC1y_1MSfCS44n7hR-Z7eU-vLtLTn&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 209.85.145.132, 2607:f8b0:4001:c10::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|209.85.145.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1484754 (1.4M) [application/octet-stream]
Saving to: ‘moesreport.pdf’

moesreport.pdf      100%[===================>]   1.42M  --.-KB/s

## Early Grade Reading Assessment Data

Early Grade Reading Assessment: received with permission from employer, RTI.
Dataset can be accessed at: data.usaid.gov

This is a dataset from the Okuu Keremet! Program's 2021 Baseline study. It has a wealth of information, ranging from G2 teacher, head teacher, and librarian interviews, classroom observations, math results, and early grade reading results. For the purpose of my capstone study, I will be using this dataset, cleaned and targeting specifically the EGRA subtasks: Oral Reading, Oral Reading Comprehension, Invented Word Reading,  Silent Reading Comprehension, and listening comprehension.


In [247]:
# let's load the data set - this is the EGRA file
!wget --no-check-certificate 'https://docs.google.com/spreadsheets/d/1Rq8nK25uEKsjwwcu0Sr1IF1oBD6N2BwbmFW5VeiGAfU/export?format=csv' -O Datareading.csv


--2025-03-29 18:28:10--  https://docs.google.com/spreadsheets/d/1Rq8nK25uEKsjwwcu0Sr1IF1oBD6N2BwbmFW5VeiGAfU/export?format=csv
Resolving docs.google.com (docs.google.com)... 142.250.125.101, 142.250.125.138, 142.250.125.100, ...
Connecting to docs.google.com (docs.google.com)|142.250.125.101|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: https://doc-0c-7s-sheets.googleusercontent.com/export/54bogvaave6cua4cdnls17ksc4/kmv24g109ml9voitcknph39cao/1743272890000/113781219181981321798/*/1Rq8nK25uEKsjwwcu0Sr1IF1oBD6N2BwbmFW5VeiGAfU?format=csv [following]
--2025-03-29 18:28:10--  https://doc-0c-7s-sheets.googleusercontent.com/export/54bogvaave6cua4cdnls17ksc4/kmv24g109ml9voitcknph39cao/1743272890000/113781219181981321798/*/1Rq8nK25uEKsjwwcu0Sr1IF1oBD6N2BwbmFW5VeiGAfU?format=csv
Resolving doc-0c-7s-sheets.googleusercontent.com (doc-0c-7s-sheets.googleusercontent.com)... 173.194.193.132, 2607:f8b0:4001:c0f::84
Connecting to doc-0c-7s-sheets.googleuser

In [248]:
egra = pd.read_csv('Datareading.csv')#let's take a look at this
egra.describe()
pd.set_option('display.max_columns', None)


<ipython-input-248-afdb31daaa64>:1: DtypeWarning:

Columns (15,18,19,20,21,22,23,24,25,27,28,30,31,32,33,34,35,36,37,38,39,44,45,46,50,52,58,59,60,67,69,70,71,80,90,95,96,100,101,102,104,108,109,110,115,121,128,132,133,134,136,137,138,139,140,141,146,147,148,152,155,156,157,158,159,167,168,176,178,179,185,192,200,201,212,219,220,229,230,241,242,243,245,253,254,263,267,268,269,271,276,277,278,280,281,290,291,292,293,294,300,301,305,306,307,311,312,313,314,315,323,324,326,327,328,333,337,338,391,392,393,396,397,452,453,454,455,456,457,458,459,460,461,462,463,464,465,468,469,475,476,478,479,485,486,488,489,500,501,503,504,510,511,513,514,520,521,523,524,533,535,549,551,555,559,563,565,567,569,571,572,573,574,575,576,582,583,584,585,586,587,588,592,593,604,605,608,610,624,626,629,630,631,632,633,634,637,638,639,640,642,643,644,645,647,648,649,650,652,653,654,655,660,661,662,663,664,665,667,670,672,673,674,675,676,677,678,679,680,681,682,683,684,685,686,687,688,689,690,691,692,693,694,695,6

,region,district,school_code,language,language_name,urbanrural,loi,school_name,treatment,loi_name,loi_kyrgyz,loi_russian,loi_uzbek,loi_tajik,stage1,fpc1,in_G2_master,in_G4_master,n_sampled,n_g4_sampled,pop_strata1,n_g2f_sampled,n_g2m_sampled,Dup_id,g_tot,n_classes,strata,n_g4f_sampled,n_g4m_sampled,grade,n_g2_sampled,pop_g2_strata1,pop_g4_strata1,g2g4_tot,pop_g2g4_strata1,g2_tot,g4_tot,g2_classes,g4_classes,in_ht,ht_start_time,ht_end_time,ht_time_start,ht_enumerator,ht_gps_latitude,ht_gps_longitude,ht_gps_accuracy,userprofileitem1first_name,userprofileitem1last_name,sh_consent,sh_female,sh_position,sh_position_other,sh_1,sh_2_1,sh_2_2,sh_2_3,sh_2_4,sh_2_777,sh_2_888,sh_2_other,sh_3_1,sh_3_2,sh_3_3,sh_3_4,sh_3_5,sh_3_6,sh_3_8,sh_3_7,sh_3_777,sh_3_888,sh_3_other,sh_4_1,sh_4_2,sh_4_3,sh_4_4,sh_4_5,sh_4_6,sh_4_7,sh_4_777,sh_4_888,sh_4_other,sh_5,sh_6,sh_7,sh_8_1,sh_8_2,sh_8_3,sh_8_4,sh_8_5,sh_8_888,sh_9,sh_10,date_end,ht_time_end,sh_8_6,ht_year,ht_date,ht_date_char,ht_dayofweek,ht_month,ht_day,sh_minutes_assess,_merge_Master_HT,in_l,l_start_time,l_end_time,l_enumerator,l_gps_latitude,l_gps_longitude,l_gps_accuracy,l_consent,l_female,l1,l2,l3,l4,l5_1,l5_2,l5_3,l5_4,l5_888,l6,l7,l8,l9,l_time_end,l_time_start,l_year,l_date,l_date_char,l_dayofweek,l_month,l_day,l_minutes_assess,_merge_Master_Lib,n_schools_sampled,wt1,sum_wt1,scale_wt1,wt_stage1,in_t,t_start_time,t_end_time,t_time_start,t_enumerator,t_gps_latitude,t_gps_longitude,t_gps_accuracy,t_consent,t_female,tq_language,tq_language_other,tq_1,tq_2,tq_3,tq_4_1,tq_4_2,tq_4_3,tq_4_888,tq_5,tq_6_1,tq_6_2,tq_6_3,tq_6_4,tq_6_5,tq_6_6,tq_6_000,tq_6_888,tq_7,tq_8,tq_9,tq_10_1,tq_10_2,tq_10_3,tq_10_4,tq_10_5,tq_10_6,tq_10_000,tq_10_888,tq_11_1,tq_11_2,tq_11_3,tq_11_4,tq_11_777,tq_11_888,tq_11_other,tq_12_1,tq_12_2,tq_12_3,tq_12_4,tq_12_5,tq_12_888,tq_13_1,tq_13_2,tq_13_3,tq_13_4,tq_13_5,tq_13_6,tq_13_7,tq_13_000,tq_13_888,tq_14,tq_15,tq_16,tq_17_1,tq_17_2,tq_17_3,tq_17_4,tq_17_5,tq_17_6,tq_17_777,tq_17_888,tq_17_other,tq_18_1,tq_18_2,tq_18_3,tq_18_4,tq_18_777,tq_18_000,tq_18_888,tq_18_other,tq_19,tq_20_1,tq_20_2,tq_20_3,tq_20_4,tq_20_5,tq_20_6,tq_20_777,tq_20_888,tq_21,tq_21_other,tq_22,tq_23_1,tq_23_2,tq_23_3,tq_23_4,tq_23_5,tq_23_6,tq_23_7,tq_23_777,tq_23_888,tq_23_other,tq_24,tq_24_other,tq_25_1,tq_25_2,tq_25_3,tq_25_4,tq_25_5,tq_25_6,tq_25_7,tq_25_777,tq_25_888,tq_26,tq_27,tq_28,tq_29,tq_30,tq_31,tq_32,t_time_end,t_year,t_date,t_date_char,t_dayofweek,t_month,t_day,t_minutes_assess,_merge_Master_Teach,in_ci,ci_start_time,ci_end_time,ci_time_start,ci_enumerator,ci_gps_latitude,ci_gps_longitude,ci_gps_accuracy,ci_language,males_enrolled,females_enrolled,ci_1_1,ci_1_2,ci_1_3,ci_1_4,ci_1_5,ci_1_6,ci_1_7,ci_1_8,ci_1_000,ci_2,ci_3,ci_4,ci_5,ci_6,ci_7,ci_8,ci_9,ci_time_end,userprofileid,ci_year,ci_date,ci_date_char,ci_dayofweek,ci_month,ci_day,ci_minutes_assess,_merge_Master_ClassInv,strata2,stage2,fpc2,wt2,wt12,wt_stage2,in_g2,sil_read_comp_score_pcnt80,sil_read_comp_attempted_pcnt80,sil_read_comp1,sil_read_comp2,sil_read_comp3,sil_read_comp4,sil_read_comp5,sil_read_comp_score,sil_read_comp_score_pcnt,sil_read_comp_score_zero,sil_read_comp_attempted,sil_read_comp_attempted_pcnt,g2_year,g2_month,g2_date,id,female,age,g2_start_time,g2_end_time,g2_consent,cnonwpm,orf,read_comp_score_pcnt80,read_comp_attempted_pcnt80,invent_word1,invent_word2,invent_word3,invent_word4,invent_word5,invent_word6,invent_word7,invent_word8,invent_word9,invent_word10,invent_word11,invent_word12,invent_word13,invent_word14,invent_word15,invent_word16,invent_word17,invent_word18,invent_word19,invent_word20,invent_word21,invent_word22,invent_word23,invent_word24,invent_word25,invent_word26,invent_word27,invent_word28,invent_word29,invent_word30,invent_word31,invent_word32,invent_word33,invent_word34,invent_word35,invent_word36,invent_word37,invent_word38,invent_word39,invent_word40,invent_word41,invent_word42,invent_word43,invent_word44,invent_word45,invent_word46,invent_word47,invent_word48,invent_word49,invent_word50,i

In [249]:
#let's check what we have. I know there's baseline and endline data here. We want just the baseline data, prior to program interventions
egra['g2_year'].value_counts()


,count
g2_year,
2024,2692
2021,1559
2021,1023
Grade 2 Read/Math Year,1


In [250]:
#Strange. I have two identical values for 2021. what's up?
egra['g2_year'].apply(type)
egra['g2_year'].dtypes

#Looks like I have strings and integers...I want to convery this all to string


,g2_year
0,<class 'str'>
1,<class 'str'>
2,<class 'str'>
3,<class 'str'>
4,<class 'str'>
...,...
5270,<class 'int'>
5271,<class 'int'>
5272,<class 'int'>
5273,<class 'int'>


dtype('O')

In [251]:
#let's see if we can convert all our "2021"'s to a string so its the same value
egra['g2_year'] = egra['g2_year'].apply(lambda x: '2021' if str(x) == '2021' else x)


In [252]:
#let's run it again, looks like it worked!
egra['g2_year'].value_counts()


,count
g2_year,
2024,2692
2021,2582
Grade 2 Read/Math Year,1


In [253]:
#now I can finally filter to the data I want.
base = egra[egra['g2_year'] == '2021']

In [254]:
base.count()
#bingo!

,0
region,2582
district,2582
school_code,2582
language,2582
language_name,2582
...,...
invent_worditem_at_time,0
oral_readitem_at_time,0
sil_readitem_at_time,0
userprofileitem1female,0


In [255]:
base.groupby('region').size()

#Breakdown by region, and by group within the baseline study of 2500 observations.
#will need to rename this, but gives some nice quick insights as to what i'll be looking at for my roll up by region!


,0
region,
Баткенская,301
Жалал-Абадская,580
Иссык-Кульская,230
Нарынская,96
Ошская,437
Таласская,140
Чуйская,341
г. Бишкек,318
г. Ош,139


In [256]:
base

,region,district,school_code,language,language_name,urbanrural,loi,school_name,treatment,loi_name,loi_kyrgyz,loi_russian,loi_uzbek,loi_tajik,stage1,fpc1,in_G2_master,in_G4_master,n_sampled,n_g4_sampled,pop_strata1,n_g2f_sampled,n_g2m_sampled,Dup_id,g_tot,n_classes,strata,n_g4f_sampled,n_g4m_sampled,grade,n_g2_sampled,pop_g2_strata1,pop_g4_strata1,g2g4_tot,pop_g2g4_strata1,g2_tot,g4_tot,g2_classes,g4_classes,in_ht,ht_start_time,ht_end_time,ht_time_start,ht_enumerator,ht_gps_latitude,ht_gps_longitude,ht_gps_accuracy,userprofileitem1first_name,userprofileitem1last_name,sh_consent,sh_female,sh_position,sh_position_other,sh_1,sh_2_1,sh_2_2,sh_2_3,sh_2_4,sh_2_777,sh_2_888,sh_2_other,sh_3_1,sh_3_2,sh_3_3,sh_3_4,sh_3_5,sh_3_6,sh_3_8,sh_3_7,sh_3_777,sh_3_888,sh_3_other,sh_4_1,sh_4_2,sh_4_3,sh_4_4,sh_4_5,sh_4_6,sh_4_7,sh_4_777,sh_4_888,sh_4_other,sh_5,sh_6,sh_7,sh_8_1,sh_8_2,sh_8_3,sh_8_4,sh_8_5,sh_8_888,sh_9,sh_10,date_end,ht_time_end,sh_8_6,ht_year,ht_date,ht_date_char,ht_dayofweek,ht_month,ht_day,sh_minutes_assess,_merge_Master_HT,in_l,l_start_time,l_end_time,l_enumerator,l_gps_latitude,l_gps_longitude,l_gps_accuracy,l_consent,l_female,l1,l2,l3,l4,l5_1,l5_2,l5_3,l5_4,l5_888,l6,l7,l8,l9,l_time_end,l_time_start,l_year,l_date,l_date_char,l_dayofweek,l_month,l_day,l_minutes_assess,_merge_Master_Lib,n_schools_sampled,wt1,sum_wt1,scale_wt1,wt_stage1,in_t,t_start_time,t_end_time,t_time_start,t_enumerator,t_gps_latitude,t_gps_longitude,t_gps_accuracy,t_consent,t_female,tq_language,tq_language_other,tq_1,tq_2,tq_3,tq_4_1,tq_4_2,tq_4_3,tq_4_888,tq_5,tq_6_1,tq_6_2,tq_6_3,tq_6_4,tq_6_5,tq_6_6,tq_6_000,tq_6_888,tq_7,tq_8,tq_9,tq_10_1,tq_10_2,tq_10_3,tq_10_4,tq_10_5,tq_10_6,tq_10_000,tq_10_888,tq_11_1,tq_11_2,tq_11_3,tq_11_4,tq_11_777,tq_11_888,tq_11_other,tq_12_1,tq_12_2,tq_12_3,tq_12_4,tq_12_5,tq_12_888,tq_13_1,tq_13_2,tq_13_3,tq_13_4,tq_13_5,tq_13_6,tq_13_7,tq_13_000,tq_13_888,tq_14,tq_15,tq_16,tq_17_1,tq_17_2,tq_17_3,tq_17_4,tq_17_5,tq_17_6,tq_17_777,tq_17_888,tq_17_other,tq_18_1,tq_18_2,tq_18_3,tq_18_4,tq_18_777,tq_18_000,tq_18_888,tq_18_other,tq_19,tq_20_1,tq_20_2,tq_20_3,tq_20_4,tq_20_5,tq_20_6,tq_20_777,tq_20_888,tq_21,tq_21_other,tq_22,tq_23_1,tq_23_2,tq_23_3,tq_23_4,tq_23_5,tq_23_6,tq_23_7,tq_23_777,tq_23_888,tq_23_other,tq_24,tq_24_other,tq_25_1,tq_25_2,tq_25_3,tq_25_4,tq_25_5,tq_25_6,tq_25_7,tq_25_777,tq_25_888,tq_26,tq_27,tq_28,tq_29,tq_30,tq_31,tq_32,t_time_end,t_year,t_date,t_date_char,t_dayofweek,t_month,t_day,t_minutes_assess,_merge_Master_Teach,in_ci,ci_start_time,ci_end_time,ci_time_start,ci_enumerator,ci_gps_latitude,ci_gps_longitude,ci_gps_accuracy,ci_language,males_enrolled,females_enrolled,ci_1_1,ci_1_2,ci_1_3,ci_1_4,ci_1_5,ci_1_6,ci_1_7,ci_1_8,ci_1_000,ci_2,ci_3,ci_4,ci_5,ci_6,ci_7,ci_8,ci_9,ci_time_end,userprofileid,ci_year,ci_date,ci_date_char,ci_dayofweek,ci_month,ci_day,ci_minutes_assess,_merge_Master_ClassInv,strata2,stage2,fpc2,wt2,wt12,wt_stage2,in_g2,sil_read_comp_score_pcnt80,sil_read_comp_attempted_pcnt80,sil_read_comp1,sil_read_comp2,sil_read_comp3,sil_read_comp4,sil_read_comp5,sil_read_comp_score,sil_read_comp_score_pcnt,sil_read_comp_score_zero,sil_read_comp_attempted,sil_read_comp_attempted_pcnt,g2_year,g2_month,g2_date,id,female,age,g2_start_time,g2_end_time,g2_consent,cnonwpm,orf,read_comp_score_pcnt80,read_comp_attempted_pcnt80,invent_word1,invent_word2,invent_word3,invent_word4,invent_word5,invent_word6,invent_word7,invent_word8,invent_word9,invent_word10,invent_word11,invent_word12,invent_word13,invent_word14,invent_word15,invent_word16,invent_word17,invent_word18,invent_word19,invent_word20,invent_word21,invent_word22,invent_word23,invent_word24,invent_word25,invent_word26,invent_word27,invent_word28,invent_word29,invent_word30,invent_word31,invent_word32,invent_word33,invent_word34,invent_word35,invent_word36,invent_word37,invent_word38,invent_word39,invent_word40,invent_word41,invent_word42,invent_word43,invent_word44,invent_word45,invent_word46,invent_word47,invent_word48,invent_word49,invent_word50,i

##Cleaning Dataset for Regional Demographics

In [257]:
moes = read_pdf('moesreport.pdf', pages='19-20',multiple_tables=True, stream=True)


In [258]:
moes = pd.DataFrame(moes[1])


In [259]:
# Move column names into the first row
moes.loc[-1] = moes.columns  # Shift column names into row 0
moes.index = moes.index + 1  # Shift index to make space
moes = moes.sort_index().reset_index(drop=True)  # Reset index

# Rename the first column if necessary
moes = moes.rename(columns={moes.columns[0]: "Ethnicity"})



In [260]:
moes

,Ethnicity,7 037 590,570 898,1 311 007,538 384,308 348,1 460 425,273 509,1 068 702,1 145 044,361 273,Unnamed: 0
0,Total population,7 037 590,570 898,1 311 007,538 384,308 348,1 460 425,273 509,1 068 702,1 145 044,361 273,Unnamed: 0
1,including:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Kyrgyz,5 470 806,451 422,967 355,492 452,307 095,1 003 764,260 038,798 347,975 128,215 205,Batken oblast
3,Russians,277 646,2 021,5 062,27 424,88,1 254,3 257,124 640,110 439,3 461,NaN
4,Uzbeks,995 454,78 314,321 461,3 463,291,421 519,1 006,17 480,15 899,136 021,Jalal-Abad
5,Ukrainians,3 257,10,88,147,-,14,55,1 715,1 199,29,oblast
6,Germans,2 747,2,55,85,6,1,143,1 768,671,16,NaN
7,Tatars,11 353,479,1 027,1 112,78,479,121,3 109,4 324,624,Issyk-Kul oblast
8,Kazakhs,28 389,317,898,5 976,190,823,2 141,10 554,7 176,314,NaN
9,Armenians,494,3,124,10,-,51,1,104,194,7,NaN


In [261]:
# Define new column names
new_column_names = {
    "Ethnicity": "Ethnicity",
    "7 037 590": "Total Population",
    "570 898": "Batken",
    "1 311 007": "Jalal-Abad",
    "538 384": "Issyk-Kul",
    "308 348": "Naryn",
    "1 460 425": "Osh",
    "273 509": "Talas",
    "1 068 702": "Chui",
    "1 145 044": "Bishkek",
    "361 273": "Osh city"
}

# Rename the columns in the DataFrame
moes = moes.rename(columns=new_column_names)

# Drop the "Unnamed: 0" column if it still exists
moes = moes.drop(columns=["Unnamed: 0"], errors="ignore")

# Display the updated DataFrame
moes

,Ethnicity,Total Population,Batken,Jalal-Abad,Issyk-Kul,Naryn,Osh,Talas,Chui,Bishkek,Osh city
0,Total population,7 037 590,570 898,1 311 007,538 384,308 348,1 460 425,273 509,1 068 702,1 145 044,361 273
1,including:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Kyrgyz,5 470 806,451 422,967 355,492 452,307 095,1 003 764,260 038,798 347,975 128,215 205
3,Russians,277 646,2 021,5 062,27 424,88,1 254,3 257,124 640,110 439,3 461
4,Uzbeks,995 454,78 314,321 461,3 463,291,421 519,1 006,17 480,15 899,136 021
5,Ukrainians,3 257,10,88,147,-,14,55,1 715,1 199,29
6,Germans,2 747,2,55,85,6,1,143,1 768,671,16
7,Tatars,11 353,479,1 027,1 112,78,479,121,3 109,4 324,624
8,Kazakhs,28 389,317,898,5 976,190,823,2 141,10 554,7 176,314
9,Armenians,494,3,124,10,-,51,1,104,194,7


In [262]:
moes.drop([1,10,16,22,28,30], inplace=True)
# Define specific ethnicity name replacements
ethnicity_replacements = {
    "nationalities": "Other nationalities",
    "and pakistan": "Indian and Pakistani",
    # Add more replacements as needed
}

# Apply the replacements
moes["Ethnicity"] = moes["Ethnicity"].replace(ethnicity_replacements)


In [263]:
for col in moes.columns[1:]:  # Skip 'Ethnicity' column
    moes[col] = moes[col].astype(str).str.replace(" ", "").str.replace("-", "0") #replace '-' with 0 for missing values. you may want to use NaN instead
    moes[col] = pd.to_numeric(moes[col], errors='coerce').fillna(0).astype(int) # Convert to numeric, handle errors, fill NaN with 0, then to int


In [264]:
moes['Ethnicity'] = moes['Ethnicity'].str.replace('Total', 'Sum') #this will do it for any ethnicity with 'Total' in the name
moes

,Ethnicity,Total Population,Batken,Jalal-Abad,Issyk-Kul,Naryn,Osh,Talas,Chui,Bishkek,Osh city
0,Sum population,7037590,570898,1311007,538384,308348,1460425,273509,1068702,1145044,361273
2,Kyrgyz,5470806,451422,967355,492452,307095,1003764,260038,798347,975128,215205
3,Russians,277646,2021,5062,27424,88,1254,3257,124640,110439,3461
4,Uzbeks,995454,78314,321461,3463,291,421519,1006,17480,15899,136021
5,Ukrainians,3257,10,88,147,0,14,55,1715,1199,29
6,Germans,2747,2,55,85,6,1,143,1768,671,16
7,Tatars,11353,479,1027,1112,78,479,121,3109,4324,624
8,Kazakhs,28389,317,898,5976,190,823,2141,10554,7176,314
9,Armenians,494,3,124,10,0,51,1,104,194,7
11,Tajiks,60752,36921,7153,213,23,8626,53,5208,1743,812


In [265]:
regselect = ['Sum population','Kyrgyz','Russians','Uzbeks','Tajiks']
regethnics = moes[moes['Ethnicity'].isin(regselect)]

In [266]:
regethnics
#table to provide total population, filtered for the ethnicities of interest

,Ethnicity,Total Population,Batken,Jalal-Abad,Issyk-Kul,Naryn,Osh,Talas,Chui,Bishkek,Osh city
0,Sum population,7037590,570898,1311007,538384,308348,1460425,273509,1068702,1145044,361273
2,Kyrgyz,5470806,451422,967355,492452,307095,1003764,260038,798347,975128,215205
3,Russians,277646,2021,5062,27424,88,1254,3257,124640,110439,3461
4,Uzbeks,995454,78314,321461,3463,291,421519,1006,17480,15899,136021
11,Tajiks,60752,36921,7153,213,23,8626,53,5208,1743,812


In [267]:
regethnics = regethnics.applymap(lambda x: f"{x:,}" if isinstance(x, (int, float)) else x)
regethnics
#We are interested in the four target languages and groups by region, as well as total population


<ipython-input-267-5a03fe27d567>:1: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



,Ethnicity,Total Population,Batken,Jalal-Abad,Issyk-Kul,Naryn,Osh,Talas,Chui,Bishkek,Osh city
0,Sum population,"7,037,590","570,898","1,311,007","538,384","308,348","1,460,425","273,509","1,068,702","1,145,044","361,273"
2,Kyrgyz,"5,470,806","451,422","967,355","492,452","307,095","1,003,764","260,038","798,347","975,128","215,205"
3,Russians,"277,646","2,021","5,062","27,424",88,"1,254","3,257","124,640","110,439","3,461"
4,Uzbeks,"995,454","78,314","321,461","3,463",291,"421,519","1,006","17,480","15,899","136,021"
11,Tajiks,"60,752","36,921","7,153",213,23,"8,626",53,"5,208","1,743",812


Remaining is to download the report, remove the indexes, and add a title "Regional population breakout by region and demographics". This will then be transferred over to the appendix.

In [268]:
regethnics.to_csv('moesreport.csv')
files.download('moesreport.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Cleaning EGRA Dataset

In [269]:
base #Cleaning dataset
pd.set_option('display.max_columns', None)


,region,district,school_code,language,language_name,urbanrural,loi,school_name,treatment,loi_name,loi_kyrgyz,loi_russian,loi_uzbek,loi_tajik,stage1,fpc1,in_G2_master,in_G4_master,n_sampled,n_g4_sampled,pop_strata1,n_g2f_sampled,n_g2m_sampled,Dup_id,g_tot,n_classes,strata,n_g4f_sampled,n_g4m_sampled,grade,n_g2_sampled,pop_g2_strata1,pop_g4_strata1,g2g4_tot,pop_g2g4_strata1,g2_tot,g4_tot,g2_classes,g4_classes,in_ht,ht_start_time,ht_end_time,ht_time_start,ht_enumerator,ht_gps_latitude,ht_gps_longitude,ht_gps_accuracy,userprofileitem1first_name,userprofileitem1last_name,sh_consent,sh_female,sh_position,sh_position_other,sh_1,sh_2_1,sh_2_2,sh_2_3,sh_2_4,sh_2_777,sh_2_888,sh_2_other,sh_3_1,sh_3_2,sh_3_3,sh_3_4,sh_3_5,sh_3_6,sh_3_8,sh_3_7,sh_3_777,sh_3_888,sh_3_other,sh_4_1,sh_4_2,sh_4_3,sh_4_4,sh_4_5,sh_4_6,sh_4_7,sh_4_777,sh_4_888,sh_4_other,sh_5,sh_6,sh_7,sh_8_1,sh_8_2,sh_8_3,sh_8_4,sh_8_5,sh_8_888,sh_9,sh_10,date_end,ht_time_end,sh_8_6,ht_year,ht_date,ht_date_char,ht_dayofweek,ht_month,ht_day,sh_minutes_assess,_merge_Master_HT,in_l,l_start_time,l_end_time,l_enumerator,l_gps_latitude,l_gps_longitude,l_gps_accuracy,l_consent,l_female,l1,l2,l3,l4,l5_1,l5_2,l5_3,l5_4,l5_888,l6,l7,l8,l9,l_time_end,l_time_start,l_year,l_date,l_date_char,l_dayofweek,l_month,l_day,l_minutes_assess,_merge_Master_Lib,n_schools_sampled,wt1,sum_wt1,scale_wt1,wt_stage1,in_t,t_start_time,t_end_time,t_time_start,t_enumerator,t_gps_latitude,t_gps_longitude,t_gps_accuracy,t_consent,t_female,tq_language,tq_language_other,tq_1,tq_2,tq_3,tq_4_1,tq_4_2,tq_4_3,tq_4_888,tq_5,tq_6_1,tq_6_2,tq_6_3,tq_6_4,tq_6_5,tq_6_6,tq_6_000,tq_6_888,tq_7,tq_8,tq_9,tq_10_1,tq_10_2,tq_10_3,tq_10_4,tq_10_5,tq_10_6,tq_10_000,tq_10_888,tq_11_1,tq_11_2,tq_11_3,tq_11_4,tq_11_777,tq_11_888,tq_11_other,tq_12_1,tq_12_2,tq_12_3,tq_12_4,tq_12_5,tq_12_888,tq_13_1,tq_13_2,tq_13_3,tq_13_4,tq_13_5,tq_13_6,tq_13_7,tq_13_000,tq_13_888,tq_14,tq_15,tq_16,tq_17_1,tq_17_2,tq_17_3,tq_17_4,tq_17_5,tq_17_6,tq_17_777,tq_17_888,tq_17_other,tq_18_1,tq_18_2,tq_18_3,tq_18_4,tq_18_777,tq_18_000,tq_18_888,tq_18_other,tq_19,tq_20_1,tq_20_2,tq_20_3,tq_20_4,tq_20_5,tq_20_6,tq_20_777,tq_20_888,tq_21,tq_21_other,tq_22,tq_23_1,tq_23_2,tq_23_3,tq_23_4,tq_23_5,tq_23_6,tq_23_7,tq_23_777,tq_23_888,tq_23_other,tq_24,tq_24_other,tq_25_1,tq_25_2,tq_25_3,tq_25_4,tq_25_5,tq_25_6,tq_25_7,tq_25_777,tq_25_888,tq_26,tq_27,tq_28,tq_29,tq_30,tq_31,tq_32,t_time_end,t_year,t_date,t_date_char,t_dayofweek,t_month,t_day,t_minutes_assess,_merge_Master_Teach,in_ci,ci_start_time,ci_end_time,ci_time_start,ci_enumerator,ci_gps_latitude,ci_gps_longitude,ci_gps_accuracy,ci_language,males_enrolled,females_enrolled,ci_1_1,ci_1_2,ci_1_3,ci_1_4,ci_1_5,ci_1_6,ci_1_7,ci_1_8,ci_1_000,ci_2,ci_3,ci_4,ci_5,ci_6,ci_7,ci_8,ci_9,ci_time_end,userprofileid,ci_year,ci_date,ci_date_char,ci_dayofweek,ci_month,ci_day,ci_minutes_assess,_merge_Master_ClassInv,strata2,stage2,fpc2,wt2,wt12,wt_stage2,in_g2,sil_read_comp_score_pcnt80,sil_read_comp_attempted_pcnt80,sil_read_comp1,sil_read_comp2,sil_read_comp3,sil_read_comp4,sil_read_comp5,sil_read_comp_score,sil_read_comp_score_pcnt,sil_read_comp_score_zero,sil_read_comp_attempted,sil_read_comp_attempted_pcnt,g2_year,g2_month,g2_date,id,female,age,g2_start_time,g2_end_time,g2_consent,cnonwpm,orf,read_comp_score_pcnt80,read_comp_attempted_pcnt80,invent_word1,invent_word2,invent_word3,invent_word4,invent_word5,invent_word6,invent_word7,invent_word8,invent_word9,invent_word10,invent_word11,invent_word12,invent_word13,invent_word14,invent_word15,invent_word16,invent_word17,invent_word18,invent_word19,invent_word20,invent_word21,invent_word22,invent_word23,invent_word24,invent_word25,invent_word26,invent_word27,invent_word28,invent_word29,invent_word30,invent_word31,invent_word32,invent_word33,invent_word34,invent_word35,invent_word36,invent_word37,invent_word38,invent_word39,invent_word40,invent_word41,invent_word42,invent_word43,invent_word44,invent_word45,invent_word46,invent_word47,invent_word48,invent_word49,invent_word50,i

In [271]:
#replace/map/recode
base['region'].replace({
    'Ошская': 'Osh',
    'г. Ош' : 'Osh_city',
    'Чуйская':'Chui',
    'Таласская':'Talas',
    'Иссык-Кульская':'Issyk-Kul',
    'г. Бишкек':'Bishkek',
    'Жалал-Абадская': 'Jalal-Abad',
    'Нарынская':'Naryn',
    'Баткенская':'Batken'}, inplace=True)
base.head(3)# names fixed,


<ipython-input-271-73aea0ebcfff>:2: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.



<ipython-input-271-73aea0ebcfff>:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,region,district,school_code,language,language_name,urbanrural,loi,school_name,treatment,loi_name,loi_kyrgyz,loi_russian,loi_uzbek,loi_tajik,stage1,fpc1,in_G2_master,in_G4_master,n_sampled,n_g4_sampled,pop_strata1,n_g2f_sampled,n_g2m_sampled,Dup_id,g_tot,n_classes,strata,n_g4f_sampled,n_g4m_sampled,grade,n_g2_sampled,pop_g2_strata1,pop_g4_strata1,g2g4_tot,pop_g2g4_strata1,g2_tot,g4_tot,g2_classes,g4_classes,in_ht,ht_start_time,ht_end_time,ht_time_start,ht_enumerator,ht_gps_latitude,ht_gps_longitude,ht_gps_accuracy,userprofileitem1first_name,userprofileitem1last_name,sh_consent,sh_female,sh_position,sh_position_other,sh_1,sh_2_1,sh_2_2,sh_2_3,sh_2_4,sh_2_777,sh_2_888,sh_2_other,sh_3_1,sh_3_2,sh_3_3,sh_3_4,sh_3_5,sh_3_6,sh_3_8,sh_3_7,sh_3_777,sh_3_888,sh_3_other,sh_4_1,sh_4_2,sh_4_3,sh_4_4,sh_4_5,sh_4_6,sh_4_7,sh_4_777,sh_4_888,sh_4_other,sh_5,sh_6,sh_7,sh_8_1,sh_8_2,sh_8_3,sh_8_4,sh_8_5,sh_8_888,sh_9,sh_10,date_end,ht_time_end,sh_8_6,ht_year,ht_date,ht_date_char,ht_dayofweek,ht_month,ht_day,sh_minutes_assess,_merge_Master_HT,in_l,l_start_time,l_end_time,l_enumerator,l_gps_latitude,l_gps_longitude,l_gps_accuracy,l_consent,l_female,l1,l2,l3,l4,l5_1,l5_2,l5_3,l5_4,l5_888,l6,l7,l8,l9,l_time_end,l_time_start,l_year,l_date,l_date_char,l_dayofweek,l_month,l_day,l_minutes_assess,_merge_Master_Lib,n_schools_sampled,wt1,sum_wt1,scale_wt1,wt_stage1,in_t,t_start_time,t_end_time,t_time_start,t_enumerator,t_gps_latitude,t_gps_longitude,t_gps_accuracy,t_consent,t_female,tq_language,tq_language_other,tq_1,tq_2,tq_3,tq_4_1,tq_4_2,tq_4_3,tq_4_888,tq_5,tq_6_1,tq_6_2,tq_6_3,tq_6_4,tq_6_5,tq_6_6,tq_6_000,tq_6_888,tq_7,tq_8,tq_9,tq_10_1,tq_10_2,tq_10_3,tq_10_4,tq_10_5,tq_10_6,tq_10_000,tq_10_888,tq_11_1,tq_11_2,tq_11_3,tq_11_4,tq_11_777,tq_11_888,tq_11_other,tq_12_1,tq_12_2,tq_12_3,tq_12_4,tq_12_5,tq_12_888,tq_13_1,tq_13_2,tq_13_3,tq_13_4,tq_13_5,tq_13_6,tq_13_7,tq_13_000,tq_13_888,tq_14,tq_15,tq_16,tq_17_1,tq_17_2,tq_17_3,tq_17_4,tq_17_5,tq_17_6,tq_17_777,tq_17_888,tq_17_other,tq_18_1,tq_18_2,tq_18_3,tq_18_4,tq_18_777,tq_18_000,tq_18_888,tq_18_other,tq_19,tq_20_1,tq_20_2,tq_20_3,tq_20_4,tq_20_5,tq_20_6,tq_20_777,tq_20_888,tq_21,tq_21_other,tq_22,tq_23_1,tq_23_2,tq_23_3,tq_23_4,tq_23_5,tq_23_6,tq_23_7,tq_23_777,tq_23_888,tq_23_other,tq_24,tq_24_other,tq_25_1,tq_25_2,tq_25_3,tq_25_4,tq_25_5,tq_25_6,tq_25_7,tq_25_777,tq_25_888,tq_26,tq_27,tq_28,tq_29,tq_30,tq_31,tq_32,t_time_end,t_year,t_date,t_date_char,t_dayofweek,t_month,t_day,t_minutes_assess,_merge_Master_Teach,in_ci,ci_start_time,ci_end_time,ci_time_start,ci_enumerator,ci_gps_latitude,ci_gps_longitude,ci_gps_accuracy,ci_language,males_enrolled,females_enrolled,ci_1_1,ci_1_2,ci_1_3,ci_1_4,ci_1_5,ci_1_6,ci_1_7,ci_1_8,ci_1_000,ci_2,ci_3,ci_4,ci_5,ci_6,ci_7,ci_8,ci_9,ci_time_end,userprofileid,ci_year,ci_date,ci_date_char,ci_dayofweek,ci_month,ci_day,ci_minutes_assess,_merge_Master_ClassInv,strata2,stage2,fpc2,wt2,wt12,wt_stage2,in_g2,sil_read_comp_score_pcnt80,sil_read_comp_attempted_pcnt80,sil_read_comp1,sil_read_comp2,sil_read_comp3,sil_read_comp4,sil_read_comp5,sil_read_comp_score,sil_read_comp_score_pcnt,sil_read_comp_score_zero,sil_read_comp_attempted,sil_read_comp_attempted_pcnt,g2_year,g2_month,g2_date,id,female,age,g2_start_time,g2_end_time,g2_consent,cnonwpm,orf,read_comp_score_pcnt80,read_comp_attempted_pcnt80,invent_word1,invent_word2,invent_word3,invent_word4,invent_word5,invent_word6,invent_word7,invent_word8,invent_word9,invent_word10,invent_word11,invent_word12,invent_word13,invent_word14,invent_word15,invent_word16,invent_word17,invent_word18,invent_word19,invent_word20,invent_word21,invent_word22,invent_word23,invent_word24,invent_word25,invent_word26,invent_word27,invent_word28,invent_word29,invent_word30,invent_word31,invent_word32,invent_word33,invent_word34,invent_word35,invent_word36,invent_word37,invent_word38,invent_word39,invent_word40,invent_word41,invent_word42,invent_word43,invent_word44,invent_word45,invent_word46,invent_word47,invent_word48,invent_word49,invent_word50,i

In [272]:
#Let's get this ordered alphabetically
base.sort_values(by='region',ascending=True,inplace=True)

<ipython-input-272-ce4a05c3bb33>:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [273]:
base['oral_read_score_pcnt'] = base['oral_read_score_pcnt'].fillna(0).round().astype(int)
base['oral_read_score_pcnt'].value_counts('')


<ipython-input-273-0d27f7ab883d>:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,count
oral_read_score_pcnt,
100,424
98,226
96,120
70,58
94,50
...,...
69,3
31,3
56,2


oral_read_score_pcnt	- DONE
g2_math_overall_score_pcnt - DONE

Now let's do some coding to clean up these remaining items that either could have NaN or odd numbers. let's go one by one.

oral_read_attempted_pcnt
read_comp_score_pcnt
word_score
word_score_pcnt
invent_word_score_pcnt


In [274]:
base['oral_read_attempted_pcnt'].value_counts('')
#same issue

,count
oral_read_attempted_pcnt,
100.0,571
100,399
98,205
98,134
96,134
...,...
66,1
63,1
47,1


In [275]:
base['oral_read_attempted_pcnt'] = base['oral_read_attempted_pcnt'].fillna(0).round().astype(int)
base['oral_read_attempted_pcnt'].value_counts()

<ipython-input-275-d097bc5717a9>:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,count
oral_read_attempted_pcnt,
100,970
98,339
96,202
97,116
95,115
...,...
29,1
45,1
49,1


In [276]:
base['read_comp_score_pcnt'].value_counts()


,count
read_comp_score_pcnt,
100.0,290
20,265
80.0,265
40.0,262
60,251
0.0,225
80,197
20,179
100,174


In [277]:
base['read_comp_score_pcnt'] = base['read_comp_score_pcnt'].fillna(0).round().astype(int)
base['read_comp_score_pcnt'].value_counts()
#looks cleaner to me!
base['list_comp_score_pcnt'] = base['list_comp_score_pcnt'].fillna(0).round().astype(int)
base['list_comp_score_pcnt'].value_counts()


<ipython-input-277-4d165f29d670>:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,count
read_comp_score_pcnt,
100,464
80,462
20,444
60,419
40,405
0,388


<ipython-input-277-4d165f29d670>:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,count
list_comp_score_pcnt,
100,661
80,589
60,517
40,343
0,257
20,215


In [278]:
base['word_score'].value_counts()
#what is going on here.
base['word_score'].dtype

,count
word_score,
4,308
3,288
6,274
5,272
2,206
3,196
2,186
1,173
4,172


dtype('O')

In [279]:
base['word_score'] = base['word_score'].fillna(0).round().astype(int)
base['word_score'].value_counts() #better!

<ipython-input-279-b759158afdb2>:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,count
word_score,
3,484
4,480
5,443
6,401
2,392
1,315
0,67


In [280]:
base['word_score_pcnt'].value_counts()

,count
word_score_pcnt,
67,308
50,288
100,274
83,272
33,206
50,196
33,186
17,173
67,172


In [281]:
base['word_score_pcnt'] = base['word_score_pcnt'].fillna(0).round().astype(int)
base['word_score_pcnt'].value_counts() #much cleaner!

<ipython-input-281-6d9ff4f1bd4b>:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,count
word_score_pcnt,
50,484
67,480
83,443
100,401
33,392
17,315
0,67


In [282]:
base['invent_word_score_pcnt'].value_counts()

,count
invent_word_score_pcnt,
46.0,52
62,50
94,46
48.0,46
54,45
...,...
10,8
2,7
4,7


In [283]:
base['invent_word_score_pcnt'] = base['invent_word_score_pcnt'].fillna(0).round().astype(int)
base['invent_word_score_pcnt'].value_counts() #fixed!

<ipython-input-283-0719fad180d7>:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,count
invent_word_score_pcnt,
46,78
64,76
56,75
60,75
62,73
54,72
98,70
52,69
44,68


In [285]:
 #subset
 base = base[[
    'region',
    'district',
    'language',
    'urbanrural',
    'female',
    'age',
    'g2_year',
    'oral_read_score_pcnt',
    'oral_read_attempted_pcnt',
    'read_comp_score_pcnt',
    'word_score',
    'word_score_pcnt',
    'invent_word_score_pcnt',
    'list_comp_score_pcnt',
    ]]
base

,region,district,language,urbanrural,female,age,g2_year,oral_read_score_pcnt,oral_read_attempted_pcnt,read_comp_score_pcnt,word_score,word_score_pcnt,invent_word_score_pcnt,list_comp_score_pcnt,sq3_2,sq3_3,sq3_4,sq3_5,sq3_6,sq3_7,sq3_777,sq3_other
385,Batken,Кадамжайский,Kyrgyz,Rural,Female,8,2021,98,98,100,2,33,96,40,0,0,0,0,0,0,0,NaN
1136,Batken,Баткенский,Kyrgyz,Rural,Female,9,2021,46,100,40,2,33,48,60,0,0,0,0.0,0.0,0.0,0,NaN
1138,Batken,Баткенский,Kyrgyz,Rural,Female,9,2021,100,100,100,3,50,46,80,0,0,0,0.0,0.0,0.0,0,NaN
272,Batken,Кадамжайский,Kyrgyz,Rural,Female,8,2021,100,100,100,2,33,96,100,0,0,0,0,0,0,0,NaN
1140,Batken,Баткенский,Kyrgyz,Rural,Female,8,2021,52,94,40,4,67,40,40,0,0,0,0.0,0.0,0.0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
794,Talas,Бакай-Атинский,Russian,Rural,Female,9,2021,43,70,20,1,17,44,0,0,0,0,0,0,0,0,NaN
1642,Talas,Кара-Бууринский,Kyrgyz,Rural,Male,8,2021,38,100,20,6,100,36,60,0,0,0,0.0,0.0,0.0,0,NaN
799,Talas,Бакай-Атинский,Russian,Rural,Male,7,2021,70,90,20,5,83,70,40,Russian,0,0,0,0,0,0,NaN
778,Talas,Кара-Бууринский,Russian,Rural,Female,7,2021,85,92,80,4,67,52,80,Russian,0,0,0,0,0,0,NaN


# EGRA Initial Tables and Graphs

In [ ]:
baselinecount = base.groupby(by=['region','language']).size().rename('count')
baselinecount



In [ ]:
baselinemean = base.pivot_table(index=
 ['region', 'language'],
 values=[
     'oral_read_score_pcnt',
     'read_comp_score_pcnt',
     'word_score_pcnt',
     'invent_word_score_pcnt',
     'list_comp_score_pcnt'],
aggfunc='mean')
baselinemean.round(2)

In [ ]:
# Define the subtask (column name)
subtask = "oral_read_score_pcnt"

# Filter dataframe for only this subtask
oralread = base[["region", subtask]].dropna()  # Remove missing values

# violin plot
fig = px.violin(oralread, x="region", y=subtask, color="region",
                box=True,  # to add mini box plot
                points="all",  # show every datapoint distributed
                title=f"Score Distribution for {subtask} Across Regions")

# Customize appearance
fig.update_layout(
    xaxis_title="Region",
    yaxis_title="Score",
    legend_title="Region",
    paper_bgcolor="white",
    font=dict(color="black")
)


In [ ]:
# Define the subtask (column name)
subtask2 = "read_comp_score_pcnt"

# Filter dataframe for only this subtask
readcomp = base[["region", subtask2]].dropna()

# Create the violin plot
fig = px.violin(readcomp, x="region", y=subtask2, color="region",
                box=True,
                points="all",
                title=f"Score Distribution for {subtask2} Across Regions")

# Customize appearance
fig.update_layout(
    xaxis_title="Region",
    yaxis_title="Score",
    legend_title="Region",
    paper_bgcolor="white",
    font=dict(color="black")
)


In [ ]:
# Define the subtask (column name)
subtask3 = "word_score_pcnt"

# Filter dataframe for only this subtask
wordscore = base[["region", subtask3]].dropna()  # Remove missing values

# Create the violin plot
fig = px.violin(wordscore, x="region", y=subtask3, color="region",
                box=True,  # Adds a mini box plot inside
                points="all",  # Show all data points
                title=f"Score Distribution for {subtask3} Across Regions")

# Customize appearance
fig.update_layout(
    xaxis_title="Region",
    yaxis_title="Score",
    legend_title="Region",
    paper_bgcolor="white",
    font=dict(color="black")
)


In [ ]:
# Define the subtask (column name)
subtask4 = "invent_word_score_pcnt"

# Filter dataframe for only this subtask
inventwordscore = base[["region", subtask4]].dropna()  # Remove missing values

# Create the violin plot
fig = px.violin(inventwordscore, x="region", y=subtask4, color="region",
                box=True,  # Adds a mini box plot inside
                points="all",  # Show all data points
                title=f"Score Distribution for {subtask4} Across Regions")

# Customize appearance
fig.update_layout(
    xaxis_title="Region",
    yaxis_title="Score",
    legend_title="Region",
    paper_bgcolor="white",
    font=dict(color="black")
)



In [ ]:
# Define the subtask (column name)
subtask5 = 'list_comp_score_pcnt'


# Filter dataframe for only this subtask
wordscore = base[["region", subtask5]].dropna()  # Remove missing values

# Create the violin plot
fig = px.violin(wordscore, x="region", y=subtask5, color="region",
                box=True,  # Adds a mini box plot inside
                points="all",  # Show all data points
                title=f"Score Distribution for {subtask5} Across Regions")

# Customize appearance
fig.update_layout(
    xaxis_title="Region",
    yaxis_title="Score",
    legend_title="Region",
    paper_bgcolor="white",
    font=dict(color="black")
)

**Oral Reading Score**
We can see that Bishkek has one heck of a tail at the upper end, almost looks like a glassblowers pipe! And the box plot is super tight!

Other regions have similar candles, with the distribution towards the top.
The interesting outliers here appear to be Jalal-Abad, Osh, and Osh City. These three , in reviewing the box plot, have a much wider range and we can see don't have the "tail" appearing at the end of the violin chart. Definitely something up with these three regions as a whole!

**Invented Word**
Aside from Bishkek, we're seeing a pretty standard distribution across the board.

**Reading Comp Score Review**
Reading comprehension followed the story/oral reading score.
If students did not fail to read the first line of words, they would be asked questions. If they did not reach certain parts of the story in time, they might not be asked certain questions.
We would expect, as a result, the results might be a bit smoother as some students wouldn't get as far along to receive the next reading comprehension question.

Osh, Talas, and Jalal Abad show a strong concentration and grouping at the bottom of the violin chart. Osh especially is showing some interesting, and perhaps troubling, results. The other regions appear to have a more normal distribution in comparison.

**Word Score Pcnt**
This subtask involved silent reading comprehension.
We're seeing similar patterns to the prior two. However, a really, really interesting note is that the students in Chui seemed to have done a bit better, with a wider distribution than all the other regions, even Bishkek, which did an amazing job on the oral reading and oral reading comprehension questions compared to the other regions. Interesting!


#Statistics

##Two Way ANOVA - between region and language

To summarize the tests run below; a two way ANOVA run on each subtask shows that 1. Region is significant 2. Language is significant. and 3, and most importantly: the interaction between region and language is significant, and differs by region. This is true for all subtasks.

This means I need to follow up digging into each region to see where language is, and isn't, significant.

###Oral Reading Score Two Way ANOVA

In [ ]:
# Fit the two-way ANOVA model
model = smf.ols('oral_read_score_pcnt ~ C(region) + C(language) + C(region):C(language)', data=base).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

# Print the ANOVA table
print(anova_table)

###Reading Comprehension Two Way ANOVA

In [ ]:
# Fit the two-way ANOVA model
model = smf.ols('read_comp_score_pcnt ~ C(region) + C(language) + C(region):C(language)', data=base).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

# Print the ANOVA table
print(anova_table)

###Silent Reading Comprehension Two Way ANOVA

In [ ]:
# Fit the two-way ANOVA model
model = smf.ols('word_score_pcnt ~ C(region) + C(language) + C(region):C(language)', data=base).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

# Print the ANOVA table
print(anova_table)

###Invented Word Score Two Way ANOVA

In [ ]:
# Fit the two-way ANOVA model
model = smf.ols('invent_word_score_pcnt ~ C(region) + C(language) + C(region):C(language)', data=base).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

# Print the ANOVA table
print(anova_table)

###List Comp Score Two Way ANOVA

In [ ]:
# Fit the two-way ANOVA model
model = smf.ols('list_comp_score_pcnt ~ C(region) + C(language) + C(region):C(language)', data=base).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

# Print the ANOVA table
print(anova_table)

##ANOVA - one way test - Language within a region by subtask

###Oral Reading Score Score - between languages within a region

In [ ]:
# List of regions
regions = base['region'].unique()

# Loop through regions and run ANOVA for each region
for region in regions:
    subset = base[base['region'] == region]
    model = ols('oral_read_score_pcnt ~ C(language)', data=subset).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"ANOVA for Region: {region}")
    print(anova_table)
    print("\n")

Takeaways of ANOVA test within regions for the oral reading  subtask score - did language matter within the region?

1. Batken - significant as p <= .05
2. Bishkek - significant as p <= .05
3. Chui - not significant as p > .05
4. Issyk-Kul - not significant as p > .05
5. Jalal-Abad - significant as p <= .05
6. Naryn - not significant as p >= .05
7. Osh - significant as p <= .05
8. Osh City - significant as p <= .05
9. Talas - not significant as p > .05


###Reading Comprehension - between languages within a region

In [ ]:
# List of regions
regions = base['region'].unique()

# Loop through regions and run ANOVA for each region
for region in regions:
    subset = base[base['region'] == region]
    model = ols('read_comp_score_pcnt ~ C(language)', data=subset).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"ANOVA for Region: {region}")
    print(anova_table)
    print("\n")

Oral Reading Comprehension  subtask, significance of language within regions
1. Batken - significant as p <= .05
2. Bishkek -  significant as p >= .05
3. Chui -  significant as p > .05
4. Issyk-Kul - not significant as p > .05
5. Jalal-Abad - significant as p <= .05
6. Naryn - not significant as p >= .05
7. Osh - significant as p <= .05
8. Osh City - not significant as p >= .05
9. Talas - not significant as p > .05

###Silent Reading Comprehension - between languages within a region

In [ ]:
# List of regions
regions = base['region'].unique()

# Loop through regions and run ANOVA for each region
for region in regions:
    subset = base[base['region'] == region]
    model = ols('word_score_pcnt ~ C(language)', data=subset).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"ANOVA for Region: {region}")
    print(anova_table)
    print("\n")

Word Score Percent subtask, significance of language within regions
1. Batken - significant as p <= .05
2. Bishkek - not significant as p >= .05
3. Chui -  significant as p > .05
4. Issyk-Kul - not significant as p > .05
5. Jalal-Abad - significant as p <= .05
6. Naryn - not significant as p >= .05
7. Osh - significant as p <= .05
8. Osh City - significant as p <= .05
9. Talas - not significant as p > .05

###Invented Word Score - between languages within a region

In [ ]:
# List of regions
regions = base['region'].unique()

# Loop through regions and run ANOVA for each region
for region in regions:
    subset = base[base['region'] == region]
    model = ols('invent_word_score_pcnt ~ C(language)', data=subset).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"ANOVA for Region: {region}")
    print(anova_table)
    print("\n")

Takeaways of ANOVA test within regions for the invented word subtask score - did language matter within the region?

1. Batken - significant as p <= .05
2. Bishkek - significant as p <= .05
3. Chui - not significant as p > .05
4. Issyk-Kul - not significant as p > .05
5. Jalal-Abad - significant as p <= .05
6. Naryn - not significant as p >= .05
7. Osh - significant as p <= .05
8. Osh City - significant as p <= .05
9. Talas - not significant as p > .05


###List Comp Score - between languages within a region

In [ ]:
# List of regions
regions = base['region'].unique()

# Loop through regions and run ANOVA for each region
for region in regions:
    subset = base[base['region'] == region]
    model = ols('list_comp_score_pcnt ~ C(language)', data=subset).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"ANOVA for Region: {region}")
    print(anova_table)
    print("\n")

##Tukey Test - One Way Significant Results

Significance of Language *within* a region
1. Oral Reading: Batken, Bishkek, Jalal-Abad, Osh, Osh City
2. Reading Comprehension: Batken, Bishkek, Chui, Jalal-Abad, Osh, Osh City
3. Silent Reading: Batken, Bishkek, Chui, Jalal-Abad, Osh, Osh City, Talas
4. Invented Word: Batken, Bishkek, Jalal-Abad, Osh, Osh City
5. Listening Comp:Batken, Bishkek, Chui, Jalal-Abad, Osh, Osh City, Talas

###Oral Reading Tukey Test, Language within region

In [ ]:
oralreadonewayanova = ['Batken', 'Bishkek', 'Jalal-Abad', 'Osh', 'Osh_city']

# Loop through each region and perform Tukey HSD test for language comparisons within each region
for region in oralreadonewayanova:
    # Filter the data for the current region
    subset = base[base['region'] == region]

    # Run Tukey HSD Test within the region for language differences
    tukey = pairwise_tukeyhsd(subset['oral_read_score_pcnt'], subset['language'], alpha=0.05)

    # Print the results for each region
    print(f"Tukey HSD Results for Region: {region}")
    print(tukey.summary())
    print("\n")

###Reading Comp Tukey Test - Language within region

In [ ]:
readcomponewayanova = ['Batken', 'Bishkek', 'Chui','Jalal-Abad', 'Osh', 'Osh_city']

# Loop through each region and perform Tukey HSD test for language comparisons within each region
for region in readcomponewayanova:
    # Filter the data for the current region
    subset = base[base['region'] == region]

    # Run Tukey HSD Test within the region for language differences
    tukey = pairwise_tukeyhsd(subset['read_comp_score_pcnt'], subset['language'], alpha=0.05)

    # Print the results for each region
    print(f"Tukey HSD Results for Region: {region}")
    print(tukey.summary())
    print("\n")

###Silent Reading Tukey Test - Language within Region

In [ ]:
wordscoreonewayanova = ['Batken', 'Bishkek', 'Chui','Jalal-Abad', 'Osh', 'Osh_city','Talas']

# Loop through each region and perform Tukey HSD test for language comparisons within each region
for region in wordscoreonewayanova:
    # Filter the data for the current region
    subset = base[base['region'] == region]

    # Run Tukey HSD Test within the region for language differences
    tukey = pairwise_tukeyhsd(subset['word_score_pcnt'], subset['language'], alpha=0.05)

    # Print the results for each region
    print(f"Tukey HSD Results for Region: {region}")
    print(tukey.summary())
    print("\n")

###Invented Word Score - Language within Region

In [ ]:
inventwordoneway = ['Batken', 'Bishkek','Jalal-Abad', 'Osh', 'Osh_city']

# Loop through each region and perform Tukey HSD test for language comparisons within each region
for region in inventwordoneway:
    # Filter the data for the current region
    subset = base[base['region'] == region]

    # Run Tukey HSD Test within the region for language differences
    tukey = pairwise_tukeyhsd(subset['invent_word_score_pcnt'], subset['language'], alpha=0.05)

    # Print the results for each region
    print(f"Tukey HSD Results for Region: {region}")
    print(tukey.summary())
    print("\n")

###Listening Comprehension Score - Language within Region

In [ ]:
listcomponewayanova = ['Batken', 'Bishkek', 'Chui','Jalal-Abad', 'Osh', 'Osh_city','Talas']

# Loop through each region and perform Tukey HSD test for language comparisons within each region
for region in listcomponewayanova:
    # Filter the data for the current region
    subset = base[base['region'] == region]

    # Run Tukey HSD Test within the region for language differences
    tukey = pairwise_tukeyhsd(subset['word_score_pcnt'], subset['language'], alpha=0.05)

    # Print the results for each region
    print(f"Tukey HSD Results for Region: {region}")
    print(tukey.summary())
    print("\n")

##Tukey Test - Two Way Group Pairings

In [ ]:
# Create a combined category for region and language
base['Region_Language'] = base['region'] + "_" + base['language']


###Oral Reading Score Tukey Test

In [ ]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['oral_read_score_pcnt'], base['Region_Language'])

# Print results
print(tukey_results)
oraltukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])
oraltukey

In [ ]:
# Assuming 'base' is your DataFrame, and it has the columns 'oral_read_score_pcnt' and 'Region_Language'

# Filter for rows where Region_Language includes 'bishkek_russian'
oshcitydata = base[base['Region_Language'] == 'Osh_city_Russian']

# Get the other regions/languages for comparison
other_regions = base[base['Region_Language'] != 'Osh_city_Russian']

# Combine the two datasets so that 'bishkek_russian' is compared to all others
comparison_data = pd.concat([oshcitydata, other_regions])

# Run Tukey's HSD test for the combined dataset
tukey_results = pairwise_tukeyhsd(comparison_data['oral_read_score_pcnt'], comparison_data['Region_Language'])

# Convert Tukey results to a DataFrame for better visualization
tukey_df = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])

# Filter the results to show only those where 'bishkek_russian' is involved in the comparison
tukey_bishkek_russian_comparisons = tukey_df[tukey_df['group1'].str.contains('Osh_city_Russian') | tukey_df['group2'].str.contains('Osh_city_Russian')]

# Display the results
print(tukey_bishkek_russian_comparisons)


In [ ]:
# Assuming 'base' is your DataFrame, and it has the columns 'oral_read_score_pcnt' and 'Region_Language'

# Filter for rows where Region_Language includes 'bishkek_russian'
oshcity_data = base[base['Region_Language'] == 'Osh_city']

# Get the other regions/languages for comparison
other_regions = base[base['Region_Language'] != 'Bishkek_Russian']

# Combine the two datasets so that 'bishkek_russian' is compared to all others
comparison_data = pd.concat([bishkek_russian_data, other_regions])

# Run Tukey's HSD test for the combined dataset
tukey_results = pairwise_tukeyhsd(comparison_data['oral_read_score_pcnt'], comparison_data['Region_Language'])

# Convert Tukey results to a DataFrame for better visualization
tukey_df = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])

# Filter the results to show only those where 'bishkek_russian' is involved in the comparison
tukey_bishkek_russian_comparisons = tukey_df[tukey_df['group1'].str.contains('Bishkek_Russian') | tukey_df['group2'].str.contains('bishkek_russian')]

# Display the results
print(tukey_bishkek_russian_comparisons)

###Reading Comprehension Score Tukey Test

In [ ]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['read_comp_score_pcnt'], base['Region_Language'])

# Print results
readcomptukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])
readcomptukey

###Silent Reading Score Tukey Test

In [ ]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['word_score_pcnt'], base['Region_Language'])
wordscoretukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])

# Print results
print(tukey_results)
wordscoretukey

###Invented Word Score Tukey Test

In [ ]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['invent_word_score_pcnt'], base['Region_Language'])
inventtukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])

# Print results
print(tukey_results)
inventtukey

###List Comprehension Score Tukey Test

In [ ]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['list_comp_score_pcnt'], base['Region_Language'])
listcomptukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])

# Print results
print(tukey_results)
listcomptukey

##Tukey Analysis

The hypothesis was that Russian would, by and large, outperform the other languages, especially in Bishkek.

For that reason, all significant differences in any pairing with "Russian" and "Bishkek" will be reviewed.

###Original sample count by region and language, and pivot table with means by subtask

In [ ]:
baselinecount = base.groupby(by=['region','language']).size().rename('count')
baselinecount
regethnics

In [ ]:
baselinemean = base.pivot_table(index=
 ['region', 'language'],
 values=[
     'oral_read_score_pcnt',
     'read_comp_score_pcnt',
     'word_score_pcnt',
     'invent_word_score_pcnt',
     'list_comp_score_pcnt'],
aggfunc='mean')
baselinemean.round(2)

###Tukey Analysis - Two Way

In [ ]:
oraltukey[oraltukey['reject'] == True]

In [ ]:
readcomptukey[readcomptukey['reject'] == True]


In [ ]:
wordscoretukey[wordscoretukey['reject'] == True]


In [ ]:
inventtukey[inventtukey['reject'] == True]


# New Section